# Colab launcher

No logic lives here. This notebook only loads secrets, pulls the repo and runs a script.

1. Runtime > Change runtime type > GPU.
2. Add secrets (key icon, left side): `ROBOFLOW_API_KEY`, `WANDB_API_KEY`, and `GITHUB_TOKEN` if the repo is private.
3. Edit `REPO` below, then run all cells.

In [ ]:
import os
from google.colab import userdata

for k in ['ROBOFLOW_API_KEY', 'WANDB_API_KEY']:
    os.environ[k] = userdata.get(k)
print('secrets loaded')

In [ ]:
REPO = 'YOUR_USER/road-violations'
BRANCH = 'main'

try:
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    url = f'https://github.com/{REPO}.git'

!rm -rf repo
!git clone -q -b {BRANCH} {url} repo
%cd repo
!pip -q install -r requirements.txt

In [ ]:
# One experiment = one config file. Change the path to run a different one.
!python scripts/train.py configs/seatbelt_yolo.yaml

In [ ]:
# Optional: copy trained weights to Google Drive so a Colab disconnect loses nothing.
import glob, os, shutil
from google.colab import drive

drive.mount('/content/drive')
dest_dir = '/content/drive/MyDrive/road-violations/models'
os.makedirs(dest_dir, exist_ok=True)
for p in glob.glob('**/weights/best.pt', recursive=True):
    dest = os.path.join(dest_dir, p.replace('/', '_'))
    shutil.copy(p, dest)
    print('saved', dest)